## Simulate Mistral 3B as participant in code switching

### Import and process the items

In [2]:
import pandas as pd
from pathlib import Path
from openai import OpenAI
import concurrent.futures


BASE_DIR = Path.cwd().parent
INPUT  = BASE_DIR / "human_data" / "CS_Spanish_completion_alltrials_n122_2025-12-19.xlsx"
OUTPUT = BASE_DIR / "llm_data" 

df_human = pd.read_excel(INPUT)
df_human = df_human[df_human["InnerNumber"].astype(str).str.fullmatch(r"\d+")]

def build_participant_items(df: pd.DataFrame) -> dict:
    participant_items = {}
    for pid, group in df.groupby("ID", sort=False):
        participant_items[pid] = group[["Item", "Itemtype"]] \
                                      .to_dict(orient="records")
    return participant_items

participant_items = build_participant_items(df_human)

### Request LLM

In [ ]:
# initial setup
MODEL_NAME = "anthropic/claude-opus-4.6"  
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="your_api_key_here", # remove api key to pull (api key is secret)
    timeout=60,
    max_retries=3
 )

# system prompt
SYSTEM_PROMPT = """You are a Spanish-English bilingual. You are participating in a linguistic task.

In this study, you will read sentences in English, Spanish or a mixture of Spanish and English. This study is completing sentences.

You will see sentences that are missing the last word. You are asked to complete each sentence in a plausible way with the first word that comes to mind. You are asked to complete only one word.

There are three sentence types. The following is an example of an English sentence. Here you would need to complete an English word.

Martha wants to buy groceries and goes to the ...

A possible completion could be the following:
Martha wants to buy groceries and goes to the ...
supermarket.

Una oración también puede estar completamente en español. A continuación, se muestra un ejemplo. Aquí, se escribe una palabra en español para completar la oración:
Marta quiere hacer compras y va al ...
supermercado.

Una oración también puede empezar in Spanish and change to English, as in the following example. Here you would complete an English word:
Marta quiere hacer compras and goes to the ...
supermarket.

Please write down what first comes to mind. Please complete the sentence with only one word. Please complete a Spanish sentence with a Spanish word, and the English and mixed sentences with an English word. Your completion needs to be plausible and grammatically correct, but other than that, you can write anything.

Important: only return the word."""

In [4]:
# define functions to run by participant
def run_single_trial(participant_id: str, item: dict) -> dict:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            max_tokens=10,
            temperature=0.1,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": item["Item"]}
            ]
        )
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": response.choices[0].message.content.strip(),
            "status":     "success"
        }
    except Exception as e:
        return {
            "ID":         participant_id,
            "Item":       item["Item"],
            "Itemtype":   item["Itemtype"],
            "Completion": None,
            "status":     str(e)
        }

def run_participant(participant_id: str, items: list[dict],
                    max_workers: int = 5) -> list[dict]:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(run_single_trial, participant_id, item)
            for item in items
        ]
        return [f.result() for f in concurrent.futures.as_completed(futures)]

def run_all(participant_items: dict, max_workers: int = 5) -> pd.DataFrame:
    all_results = []
    participants = list(participant_items.items())
    total = len(participants)
    
    for i, (pid, items) in enumerate(participants, 1):
        print(f"Running participant {i}/{total}: {pid}")
        results = run_participant(pid, items, max_workers)
        all_results.extend(results)
        
        # checkpoint every 10 participants
        if i % 10 == 0:
            df_checkpoint = pd.DataFrame(all_results)
            checkpoint_path = OUTPUT / f"checkpoint_{i}of{total}.csv"
            df_checkpoint.to_csv(checkpoint_path, index=False)  
            print(f"  ✓ Checkpoint: {checkpoint_path.name}")
    
    return pd.DataFrame(all_results)  

In [ ]:
# Run all participants and get the results in a DataFrame
df_llm = run_all(participant_items)

### Check and save the output data

In [7]:
# check failed trial
failed = df_llm[df_llm["status"] != "success"]
if len(failed) > 0:
    print(f"⚠️ failed trial: {len(failed)}个")
    print(failed[["ID", "Item", "status"]])
else:
    print(f"✓ all success，in total of {len(df_llm)} trials")
# delete status column and save to excel
df_llm = df_llm.drop(columns=["status"])
df_llm.to_csv(OUTPUT / f"llm_{MODEL_NAME.replace('/', '-')}.csv", index=False)

✓ all success，in total of 5745 trials


### test

In [5]:
# ── use the first participant for testing ─────────────────────────────────────
first_pid = list(participant_items.keys())[0]
first_items = participant_items[first_pid]

for item in first_items[:3]:
    print(f"  {item}")

# ── only run this participant ───────────────────────────────────────
#test single participant
test_results = run_participant(first_pid, first_items)
df_test = pd.DataFrame(test_results)
print(df_test[["Itemtype", "Completion", "status"]])

  {'Item': 'William played his stereo much too', 'Itemtype': 'E'}
  {'Item': 'Está lloviendo, y me olvidé de mi coat, hat, and', 'Itemtype': 'CS'}
  {'Item': 'Our special guests should be arriving', 'Itemtype': 'E'}
   Itemtype Completion   status
0        CS   umbrella  success
1         E      soon.  success
2        CS    channel  success
3        CS      nurse  success
4         E      loud.  success
5         E    thunder  success
6        CS      beach  success
7        CS     splash  success
8         E        fog  success
9        CS   diamond.  success
10       CS      room.  success
11       CS     paint.  success
12        E   talking.  success
13       CS      music  success
14        E   bedrooms  success
15        E       fall  success
16       CS      towel  success
17        E       web.  success
18        E      short  success
19       CS      glass  success
20       CS      books  success
21        E      stamp  success
22        E     coffee  success
23       CS     